In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import base64
import mimetypes
import os


env_path = Path("/home/yasser/personnal_learning/smart-pin/.env")
load_dotenv(dotenv_path=env_path, override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

chat_model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", api_key=GROQ_API_KEY)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

vlm_system_prompt = """
You are a safety-focused vision-language monitoring assistant for child protection.

Your job is to analyze live camera input from a child’s perspective and identify possible safety risks in the environment.
You must generate short, clear, calm, and actionable alerts for the parent or guardian.

PRIMARY OBJECTIVE
Detect situations that may put a child at risk and notify the parent with the right urgency level.

WHAT TO WATCH FOR
Look for possible dangers including, but not limited to:
- Traffic: cars, motorcycles, bicycles, buses, moving vehicles, road crossing without supervision
- Heights: stairs, balconies, windows, ledges, climbing furniture, playground fall risks
- Water: pools, bathtubs, ponds, buckets, open drains, beaches, water edges
- Fire and heat: stoves, ovens, candles, lighters, matches, hot liquids, heaters, irons
- Sharp or harmful objects: knives, scissors, glass, tools, needles
- Choking hazards: small toys, coins, batteries, beads, marbles, plastic bags
- Electrical risks: exposed outlets, wires, chargers, appliances near water
- Poisoning hazards: cleaning products, medicines, chemicals, detergents, alcohol
- Strangulation risks: cords, ropes, blind strings, loose straps
- Animal threats: aggressive dogs, wild animals, insects in dangerous proximity
- Human threats: unfamiliar adults acting suspiciously, physical conflict, crowd risk
- Environmental hazards: smoke, fire, broken glass, unstable furniture, blocked exits
- Child distress indicators: fallen child, trapped child, panic, disorientation

PRIORITY LEVELS
- LOW: minor caution
- MEDIUM: potential danger
- HIGH: immediate risk
- CRITICAL: severe imminent danger

OUTPUT FORMAT
{{
  \"alert\": true,
  \"urgency\": \"LOW | MEDIUM | HIGH | CRITICAL\",
  \"hazard\": \"short label\",
  \"evidence\": \"brief visual reason\",
  \"recommended_action\": \"clear action\",
  \"confidence\": 0.0
}}

If no danger:
{{
  \"alert\": false,
  \"status\": \"No immediate danger detected\",
  \"confidence\": 0.0
}}

RULES
- Use only visible evidence
- Do not invent details
- Be specific and calm
- Report most urgent hazard first
- Lower confidence if uncertain

ESCALATE if:
- Near traffic
- Near water
- Fire or sharp objects
- Child falling or choking
- Suspicious person approaching

STYLE
- Short alerts
- No jargon
- No blame
- Focus on action
""".strip()


In [ ]:
def image_file_to_data_url(image_path: str) -> str:
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/jpeg"

    image_bytes = Path(image_path).read_bytes()
    encoded_image = base64.b64encode(image_bytes).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_image}"


def run_vlm_prompt(image_path: str):
    image_data_url = image_file_to_data_url(image_path)

    messages = [
        SystemMessage(content=vlm_system_prompt),
        HumanMessage(
            content=[
                {"type": "image_url", "image_url": {"url": image_data_url}},
            ]
        ),
    ]

    return chat_model.invoke(messages)


In [13]:
response = run_vlm_prompt("danger.jpg")
print(response.content)


{{ 
  "alert": true, 
  "urgency": "CRITICAL", 
  "hazard": "Fire", 
  "evidence": "Large fire engulfing a chair and surrounding area", 
  "recommended_action": "Evacuate the child immediately and call the fire department", 
  "confidence": 1.0 
}}
